# Graph Mining Demo: Graph Representation, Centrality, and Communities

This notebook introduces core graph mining concepts used in modern data science, network analysis, web mining, social analytics, recommendation systems, and knowledge graphs. We focus on three tightly connected ideas:

- **Graph representation**: how to encode nodes and edges for computation
- **Centrality**: how to quantify importance or influence of nodes
- **Communities**: how to discover groups of densely connected nodes

The notebook is designed as a class demo and mixes short theory explanations with executable Python examples.

#### Import required packages

In [ ]:
import math
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

print("Libraries imported successfully.")
print("NetworkX version:", nx.__version__)

## Create a Small Example Network

We begin with a small undirected graph representing interactions among entities.  
This kind of toy graph is useful for understanding graph concepts before moving to large real-world networks.

In [ ]:
G = nx.Graph()

edges = [
    ("A", "B"), ("A", "C"), ("A", "D"),
    ("B", "C"), ("B", "D"),
    ("C", "D"),
    ("D", "E"),
    ("E", "F"), ("E", "G"),
    ("F", "G"), ("F", "H"),
    ("G", "H"),
    ("H", "I"),
    ("I", "J"), ("I", "K"),
    ("J", "K")
]

G.add_edges_from(edges)

print("Number of nodes:", G.number_of_nodes())
print("Number of edges:", G.number_of_edges())
print("Nodes:", list(G.nodes()))

## Visualize the Graph

A quick plot gives intuition about local density, bridge nodes, and possible communities.

In [ ]:
plt.figure(figsize=(8, 6))
pos = nx.spring_layout(G, seed=42)
nx.draw(
    G, pos,
    with_labels=True,
    node_size=1200,
    font_size=10
)
plt.title("Example Undirected Graph")
plt.axis("off")
plt.show()

## Adjacency List Representation

In an adjacency list, each node maps to a list of its neighbors.  
This is a natural and memory-efficient representation for sparse graphs.

In [ ]:
adj_list = {node: sorted(list(G.neighbors(node))) for node in G.nodes()}
adj_list

## Adjacency Matrix Representation

The adjacency matrix is often useful for matrix-based graph analysis, spectral methods, and graph learning.

In [ ]:
adj_matrix = nx.to_pandas_adjacency(G, dtype=int)
adj_matrix

## Basic Graph Statistics

Before advanced mining, it is useful to inspect:
- degree of each node
- density
- connected components
- shortest paths and distances

### Graph Density

**Meaning:**  
Density measures how connected the graph is.

$$
\text{Density} = \frac{2E}{N(N - 1)}
$$

- $E$ = number of edges  
- $N$ = number of nodes  

👉 **Interpretation:**
- Close to **1** → very dense (many connections)  
- Close to **0** → sparse (few connections)  

**Example:**  
Graph density = 0.2909  

→ About **29% of all possible edges exist**

**Example:**

In [ ]:
degrees = dict(G.degree())
stats_df = pd.DataFrame({
    "Node": list(degrees.keys()),
    "Degree": list(degrees.values())
}).sort_values("Degree", ascending=False)

print("Graph density:", round(nx.density(G), 4))
print("Is connected?:", nx.is_connected(G))
stats_df

## Degree Centrality

Degree centrality measures how many immediate neighbors a node has, normalized by the maximum possible number of neighbors.

This is a local measure of importance and is often a strong baseline.

In [ ]:
degree_cent = nx.degree_centrality(G)
degree_df = pd.DataFrame({
    "Node": list(degree_cent.keys()),
    "DegreeCentrality": list(degree_cent.values())
}).sort_values("DegreeCentrality", ascending=False)

degree_df

## Closeness Centrality

Closeness centrality rewards nodes that can reach all other nodes with short path lengths.

A node with high closeness is often centrally located in the network.

In [ ]:
closeness_cent = nx.closeness_centrality(G)
closeness_df = pd.DataFrame({
    "Node": list(closeness_cent.keys()),
    "ClosenessCentrality": list(closeness_cent.values())
}).sort_values("ClosenessCentrality", ascending=False)

closeness_df

## Betweenness Centrality

Betweenness centrality measures how frequently a node lies on shortest paths between other pairs of nodes.

Nodes with high betweenness often act as **bridges** or **bottlenecks**.

In [ ]:
betweenness_cent = nx.betweenness_centrality(G, normalized=True)
betweenness_df = pd.DataFrame({
    "Node": list(betweenness_cent.keys()),
    "BetweennessCentrality": list(betweenness_cent.values())
}).sort_values("BetweennessCentrality", ascending=False)

betweenness_df

### Eigenvector Centrality

**Definition:**  
Eigenvector centrality assigns higher scores to nodes that are connected to other important (high-scoring) nodes.

$$
x_i = \frac{1}{\lambda} \sum_{j} A_{ij} \, x_j
$$

- $x_i$ = centrality of node $i$  
- $A_{ij}$ = adjacency matrix entry (1 if connected, else 0)  
- $\lambda$ = largest eigenvalue  

👉 **Key Idea:**  
A node is important if it is connected to other important nodes.

👉 **Interpretation:**
- High score → connected to influential nodes  
- Low score → connected to less important nodes  

👉 **Applications:**
- Web page ranking (e.g., PageRank)  
- Citation networks  
- Social network analysis  

👉 **Insight:**  
Eigenvector centrality captures **global influence**, unlike degree centrality which only considers local connections.

In [ ]:
eigenvector_cent = nx.eigenvector_centrality(G, max_iter=1000)
eigenvector_df = pd.DataFrame({
    "Node": list(eigenvector_cent.keys()),
    "EigenvectorCentrality": list(eigenvector_cent.values())
}).sort_values("EigenvectorCentrality", ascending=False)

eigenvector_df

## Compare Centrality Measures Side by Side

Different centrality measures may rank nodes differently because they capture different structural roles.

In [ ]:
centrality_compare = pd.DataFrame({"Node": list(G.nodes())})
centrality_compare["Degree"] = centrality_compare["Node"].map(degree_cent)
centrality_compare["Closeness"] = centrality_compare["Node"].map(closeness_cent)
centrality_compare["Betweenness"] = centrality_compare["Node"].map(betweenness_cent)
centrality_compare["Eigenvector"] = centrality_compare["Node"].map(eigenvector_cent)

centrality_compare = centrality_compare.round(4)
centrality_compare.sort_values("Betweenness", ascending=False)

## Visualize a Centrality Measure on the Graph

A helpful teaching strategy is to map node size to centrality.  
Here, larger nodes indicate higher betweenness centrality.

In [ ]:
sizes = [4000 * betweenness_cent[n] + 500 for n in G.nodes()]

plt.figure(figsize=(8, 6))
nx.draw(
    G, pos,
    with_labels=True,
    node_size=sizes,
    font_size=10
)
plt.title("Graph with Node Size Proportional to Betweenness Centrality")
plt.axis("off")
plt.show()

## From Graph Representation to Community Detection

The graph appears to have tightly connected local groups with a few bridging edges between them.  
This is a common sign of **community structure**.

We now apply a community detection method to partition the graph.

### Modularity-Based Community Detection

**Definition:**  
A modularity-based method detects communities by maximizing **modularity**, a measure of how well a graph is partitioned into dense groups with sparse connections between them.

---

### Modularity Formula

$$
Q = \frac{1}{2m} \sum_{i,j} \left( A_{ij} - \frac{k_i k_j}{2m} \right) \delta(c_i, c_j)
$$

- $A_{ij}$ = adjacency matrix (1 if edge exists, else 0)  
- $k_i, k_j$ = degrees of nodes $i$ and $j$  
- $m$ = total number of edges  
- $\delta(c_i, c_j)$ = 1 if nodes $i$ and $j$ are in the same communityeen-community) edges  

---

### Greedy Modularity Algorithm

- Starts with each node as its own community  
- Iteratively merges communities  
- At each step, chooses the merge that **maximizes modularity increase**

👉 Stops when no further improvement is possible

---

### Interpretation

- **High modularity (≈ 0.3–0.7):** strong community structure  
- **Low modularity (≈ 0):** weak or no clear communities  

---

### Key Insight

👉 Modularity-based methods automatically discover **natural groupings** in a network by optimizing structural quality, not predefined labels.

In [ ]:
from networkx.algorithms.community import greedy_modularity_communities

communities = list(greedy_modularity_communities(G))
community_sets = [sorted(list(c)) for c in communities]

for i, c in enumerate(community_sets, start=1):
    print(f"Community {i}: {c}")

## Assign Community Labels to Nodes

To analyze or visualize the result, we map each node to a detected community ID.

In [ ]:
node_to_comm = {}
for i, comm in enumerate(communities):
    for node in comm:
        node_to_comm[node] = i

community_df = pd.DataFrame({
    "Node": list(G.nodes()),
    "Community": [node_to_comm[n] for n in G.nodes()]
}).sort_values(["Community", "Node"])

community_df

## Visualize Communities

We now draw the graph with nodes colored by detected community membership.

In [ ]:
plt.figure(figsize=(8, 6))
node_colors = [node_to_comm[n] for n in G.nodes()]
nx.draw(
    G, pos,
    with_labels=True,
    node_size=1200,
    node_color=node_colors,
    font_size=10
)
plt.title("Detected Communities in the Graph")
plt.axis("off")
plt.show()

## Evaluate Community Quality with Modularity

**Modularity** measures whether the graph partition contains more within-community edges than expected under a random baseline.

Higher modularity often suggests a stronger community structure.

### Rule of Thumb

- **Q ≈ 0** → no meaningful community structure  
- **Q ≈ 0.3 – 0.7** → significant community structure  
- **Q > 0.7** → very strong (rare in real-world networks)

---

In [ ]:
from networkx.algorithms.community.quality import modularity

mod_value = modularity(G, communities)
print("Modularity of detected partition:", round(mod_value, 4))

### Insight

👉 A modularity of **0.5215** confirms that the detected communities:
- Are **well-formed**
- Reflect **natural groupings** in the network


## Build a Larger Synthetic Graph with Known Communities

For a more realistic demo, we generate a graph with planted community structure using a stochastic block model.  
This helps students see how community detection scales beyond toy examples.

In [ ]:
sizes = [12, 12, 12]
probs = [
    [0.45, 0.03, 0.02],
    [0.03, 0.40, 0.03],
    [0.02, 0.03, 0.42]
]

SG = nx.stochastic_block_model(sizes, probs, seed=42)
print("Synthetic graph nodes:", SG.number_of_nodes())
print("Synthetic graph edges:", SG.number_of_edges())

## Detect Communities on the Synthetic Graph

Because within-group edge probabilities are much higher than across-group probabilities, the graph should contain meaningful communities.

In [ ]:
sg_communities = list(greedy_modularity_communities(SG))
print("Number of detected communities:", len(sg_communities))

sg_mod = modularity(SG, sg_communities)
print("Modularity:", round(sg_mod, 4))

## Visualize the Synthetic Community Structure

This plot usually reveals three dense blocks, illustrating how graph mining can uncover hidden structure from connectivity alone.

In [ ]:
plt.figure(figsize=(8, 6))
sg_pos = nx.spring_layout(SG, seed=42)

sg_node_to_comm = {}
for i, comm in enumerate(sg_communities):
    for node in comm:
        sg_node_to_comm[node] = i

sg_colors = [sg_node_to_comm[n] for n in SG.nodes()]

nx.draw(
    SG, sg_pos,
    node_size=250,
    node_color=sg_colors,
    with_labels=False
)
plt.title("Synthetic Graph with Detected Communities")
plt.axis("off")
plt.show()

## Edge Density and Clustering Coefficient

Two useful structural properties of a social network are **density** and **clustering**.

- **Density** tells us how full the graph is compared to the maximum possible number of edges.
- **Average clustering coefficient** tells us how likely it is that a node's neighbors are also connected to one another.

In a social setting:
- A **high-density** graph suggests many relationships overall.
- A **high clustering coefficient** often indicates **friend circles** or local groups where “friends of friends are also friends.”

These measures help us understand whether a network is globally sparse, locally cohesive, or both.

We have already covered **Density** earlier.

### Average Clustering Coefficient

**Definition:**  
Measures how likely neighbors of a node are connected.

$$
C = \frac{1}{N} \sum_{i=1}^{N} C_i
$$

- $C_i$ = local clustering of node $i$

---

### Local Clustering Coefficient

$$
C_i = \frac{2 e_i}{k_i (k_i - 1)}
$$

- $e_i$ = edges between neighbors  
- $k_i$ = degree of node $i$  

👉 If $k_i < 2$, then $C_i = 0$

---

### Interpretation

- $C_i = 1$ → neighbors fully connected  
- $C_i = 0$ → no neighbor connections  

👉 Measures **local cohesiveness (triangles / friend-of-friend)**ss** (friend-of-friend connectivity)

In [ ]:
# Edge density and average clustering coefficient
density = nx.density(G)
avg_clustering = nx.average_clustering(G)

density_df = pd.DataFrame({
    "metric": ["edge_density", "average_clustering"],
    "value": [density, avg_clustering]
})

density_df

### Combined Insight

- **Moderate density (0.29)** → not globally dense  
- **High clustering (0.71)** → strong local groups  

## Connected Components

A **connected component** is a subset of nodes where every node can reach every other node through some path.

Why this matters in social network mining:
- If a graph has **one giant component**, then most users are part of the same broad social structure.
- If there are **many small components**, then the network is fragmented into disconnected groups.
- Isolated nodes or tiny components may indicate weak participation, incomplete data, or separate communities.

For undirected social graphs, connected components are a basic but important way to understand overall connectivity.

In [ ]:
# Connected components in the graph
components = list(nx.connected_components(G))

component_sizes_df = pd.DataFrame({
    "component_id": range(len(components)),
    "size": [len(c) for c in components]
}).sort_values("size", ascending=False).reset_index(drop=True)

component_sizes_df

## Triangle Counts and Local Cohesion

A **triangle** appears when three nodes are all connected to each other.  
In social networks, triangles are extremely meaningful because they often reflect **strong local cohesion**.

For example:
- If Alice knows Bob, and Bob knows Carol, then a triangle forms when Alice also knows Carol.
- A large number of triangles often signals **trust**, **friend-group formation**, or **shared social circles**.

Counting triangles helps reveal how strongly local neighborhoods are tied together, beyond what degree alone can show.

In [ ]:
# Triangle counts for each node
triangle_counts = nx.triangles(G)

triangle_df = pd.DataFrame({
    "node": list(triangle_counts.keys()),
    "triangle_count": list(triangle_counts.values())
}).sort_values("triangle_count", ascending=False).reset_index(drop=True)

triangle_df.head(10)

## k-Core Decomposition

A **k-core** is a maximal subgraph in which every node has at least **k connections within that subgraph**.

This is useful in social network mining because it helps identify a **stable and well-connected core** of the network:
- Peripheral users may have only a few connections.
- Core users remain strongly connected even after weaker nodes are removed.
- Higher core numbers often indicate more embedded, central, or resilient participation.

Compared with simple degree counts, k-core analysis better captures how deeply a node sits inside the network structure.

In [ ]:
# k-core / core number analysis
core_numbers = nx.core_number(G)

core_df = pd.DataFrame({
    "node": list(core_numbers.keys()),
    "core_number": list(core_numbers.values())
}).sort_values("core_number", ascending=False).reset_index(drop=True)

core_df.head(10)

## Assortativity: Do Similar Nodes Connect?

**Assortativity** measures whether nodes tend to connect to other nodes that are similar in some way.  
A common version is **degree assortativity**, which asks:

> Do highly connected users tend to connect to other highly connected users?

Interpretation:
- **Positive assortativity**: high-degree nodes tend to connect with high-degree nodes.
- **Negative assortativity**: high-degree nodes tend to connect with low-degree nodes.
- **Near zero**: little clear tendency.

In many social networks, assortativity helps us understand whether influence is concentrated among elites or spread more broadly.

In [ ]:
# Degree assortativity coefficient
assortativity_value = nx.degree_assortativity_coefficient(G)

assortativity_df = pd.DataFrame({
    "metric": ["degree_assortativity"],
    "value": [assortativity_value]
})

assortativity_df

## Summary and Data Mining Interpretation

This notebook demonstrated three major graph mining ideas:

1. **Graph representation**  
   We represented the same network as an adjacency list and an adjacency matrix.

2. **Centrality**  
   We measured node importance using degree, closeness, betweenness, and eigenvector centrality.

3. **Communities**  
   We detected densely connected groups and evaluated them using modularity.

These techniques are useful in many applications:
- social network analysis
- web link mining
- fraud detection
- recommender systems
- biological network analysis
- citation and knowledge graphs

A key lesson is that graph mining is not only about visualization. It provides quantitative tools for extracting structure, influence, and grouping patterns from relational data.

# See you next lecture!